# Assignment 2: CNN classification of BloodMNIST

**Group:** Daniel, Frederik and Jonas  
**Research question:** How accurately can a compact convolutional neural network classify eight types of peripheral blood cells in BloodMNIST, and which classes remain difficult to distinguish?

This notebook keeps the official MedMNIST training, validation and test splits separate. The validation set is used to select the model. The test set is evaluated only once after model selection.


## 1. Setup

If an import fails, uncomment and run the installation line. Google Colab already includes PyTorch.


In [ ]:
# Uncomment only if packages are missing in the current notebook environment.
# %pip install -q medmnist matplotlib scikit-learn

from copy import deepcopy
from pathlib import Path
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)


## 2. Load BloodMNIST

The notebook first looks for the dataset downloaded during Assignment 1. If it is missing, the fallback uses the official `medmnist` package to download it.


In [ ]:
DATA_PATH = Path.home() / ".medmnist" / "bloodmnist.npz"

if not DATA_PATH.exists():
    try:
        from medmnist import BloodMNIST
    except ImportError as exc:
        raise ImportError(
            "Run the installation cell above, then rerun this cell."
        ) from exc
    BloodMNIST(split="train", download=True, size=28)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"BloodMNIST was not found at {DATA_PATH}")

raw = np.load(DATA_PATH)
X_train_raw = raw["train_images"]
y_train = raw["train_labels"].reshape(-1).astype(np.int64)
X_val_raw = raw["val_images"]
y_val = raw["val_labels"].reshape(-1).astype(np.int64)
X_test_raw = raw["test_images"]
y_test = raw["test_labels"].reshape(-1).astype(np.int64)

CLASS_NAMES = [
    "Basophil",
    "Eosinophil",
    "Erythroblast",
    "Immature granulocytes",
    "Lymphocyte",
    "Monocyte",
    "Neutrophil",
    "Platelet",
]

print("Train:", X_train_raw.shape, y_train.shape)
print("Validation:", X_val_raw.shape, y_val.shape)
print("Test:", X_test_raw.shape, y_test.shape)


## 3. Materials: inspect the dataset

Use these figures in the Materials section. Discuss class imbalance and whether classes look homogeneous within a class and visually distinct between classes.


In [ ]:
split_counts = {
    "Train": np.bincount(y_train, minlength=len(CLASS_NAMES)),
    "Validation": np.bincount(y_val, minlength=len(CLASS_NAMES)),
    "Test": np.bincount(y_test, minlength=len(CLASS_NAMES)),
}

print(f"{'Class':24s} {'Train':>7s} {'Val':>7s} {'Test':>7s}")
for i, name in enumerate(CLASS_NAMES):
    print(
        f"{name:24s} {split_counts['Train'][i]:7d} "
        f"{split_counts['Validation'][i]:7d} {split_counts['Test'][i]:7d}"
    )

x = np.arange(len(CLASS_NAMES))
width = 0.26
fig, ax = plt.subplots(figsize=(12, 5))
for offset, (split_name, counts) in enumerate(split_counts.items()):
    ax.bar(x + (offset - 1) * width, counts, width, label=split_name)
ax.set_ylabel("Number of images")
ax.set_title("BloodMNIST class distribution")
ax.set_xticks(x, CLASS_NAMES, rotation=35, ha="right")
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(3, len(CLASS_NAMES), figsize=(16, 6))
for class_id, class_name in enumerate(CLASS_NAMES):
    candidates = np.flatnonzero(y_train == class_id)
    chosen = rng.choice(candidates, size=3, replace=False)
    for row, image_index in enumerate(chosen):
        axes[row, class_id].imshow(X_train_raw[image_index])
        axes[row, class_id].axis("off")
        if row == 0:
            axes[row, class_id].set_title(class_name, fontsize=9)
fig.suptitle("Example training images from each class")
fig.tight_layout()
plt.show()


## 4. Preprocessing and data loaders

Pixel values are converted from integers in `[0, 255]` to floating-point values and standardized using the training-set channel mean and standard deviation. Statistics from the validation or test set are not used during preprocessing.


In [ ]:
train_float = X_train_raw.astype(np.float32) / 255.0
channel_mean = train_float.mean(axis=(0, 1, 2))
channel_std = train_float.std(axis=(0, 1, 2))
print("Training channel mean:", channel_mean)
print("Training channel std:", channel_std)


class BloodDataset(Dataset):
    def __init__(self, images, labels, mean, std, augment=False):
        self.images = images
        self.labels = labels
        self.mean = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = torch.from_numpy(self.images[index]).permute(2, 0, 1).float() / 255.0
        if self.augment:
            # Cell orientation is not meaningful, so flips are plausible augmentations.
            if torch.rand(()) < 0.5:
                image = torch.flip(image, dims=(2,))
            if torch.rand(()) < 0.5:
                image = torch.flip(image, dims=(1,))
        image = (image - self.mean) / self.std
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return image, label


def make_loaders(batch_size=64, augment=False):
    train_dataset = BloodDataset(
        X_train_raw, y_train, channel_mean, channel_std, augment=augment
    )
    val_dataset = BloodDataset(X_val_raw, y_val, channel_mean, channel_std)
    test_dataset = BloodDataset(X_test_raw, y_test, channel_mean, channel_std)

    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(batch_size=64)
images, labels = next(iter(train_loader))
print("One batch:", images.shape, labels.shape)


## 5. CNN architecture

Convolutional layers preserve spatial structure, unlike the flattened fully connected network from Assignment 1. Small `3 × 3` kernels learn local patterns such as cell edges, textures and colour variations. Max pooling reduces spatial dimensions, batch normalization stabilizes training, and dropout provides regularization.


In [ ]:
class BloodCNN(nn.Module):
    def __init__(self, num_classes=8, base_channels=32, dropout=0.30):
        super().__init__()
        c = base_channels
        self.features = nn.Sequential(
            nn.Conv2d(3, c, kernel_size=3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(c, 2 * c, kernel_size=3, padding=1),
            nn.BatchNorm2d(2 * c),
            nn.ReLU(inplace=True),
            nn.Conv2d(2 * c, 2 * c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(2 * c, 4 * c, kernel_size=3, padding=1),
            nn.BatchNorm2d(4 * c),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(4 * c, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = BloodCNN().to(DEVICE)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Trainable parameters: {parameter_count:,}")

with torch.no_grad():
    output = model(images[:4].to(DEVICE))
print("Output shape:", output.shape)


## 6. Training and validation functions

Cross-entropy loss is appropriate for a single-label, eight-class problem. Adam updates the model parameters. Early stopping retains the epoch with the lowest validation loss.


In [ ]:
def run_epoch(model, loader, loss_function, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for inputs, targets in loader:
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(inputs)
            loss = loss_function(logits, targets)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * targets.size(0)
        total_correct += (logits.argmax(dim=1) == targets).sum().item()
        total_examples += targets.size(0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


def train_model(config):
    torch.manual_seed(SEED)
    train_loader, val_loader, _ = make_loaders(
        batch_size=config["batch_size"], augment=config["augment"]
    )
    model = BloodCNN(
        base_channels=config["base_channels"], dropout=config["dropout"]
    ).to(DEVICE)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
    best_state = None
    best_val_loss = float("inf")
    best_val_accuracy = 0.0
    best_epoch = 0
    epochs_without_improvement = 0
    start = time.time()

    for epoch in range(1, config["max_epochs"] + 1):
        train_metrics = run_epoch(model, train_loader, loss_function, optimizer)
        val_metrics = run_epoch(model, val_loader, loss_function)

        history["train_loss"].append(train_metrics["loss"])
        history["train_accuracy"].append(train_metrics["accuracy"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])

        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_metrics['loss']:.4f}, acc {train_metrics['accuracy']:.4f} | "
            f"val loss {val_metrics['loss']:.4f}, acc {val_metrics['accuracy']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss - 1e-4:
            best_val_loss = val_metrics["loss"]
            best_val_accuracy = val_metrics["accuracy"]
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= config["patience"]:
            print("Early stopping")
            break

    model.load_state_dict(best_state)
    elapsed = time.time() - start
    summary = {
        "name": config["name"],
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_accuracy": best_val_accuracy,
        "seconds": elapsed,
    }
    return model, history, summary


## 7. Hyperparameter experiments

Start with `QUICK_RUN = True` to confirm that the pipeline works. Then set it to `False` for the report experiments. The test loader is deliberately not used here.

The three experiments isolate modest changes to learning rate, dropout and data augmentation. Record every attempted configuration, not only the winner.


In [ ]:
QUICK_RUN = True  # Change to False when ready for the report experiments.

common = {
    "batch_size": 64,
    "base_channels": 32,
    "weight_decay": 1e-4,
    "patience": 5,
    "max_epochs": 4 if QUICK_RUN else 25,
}

experiments = [
    {
        **common,
        "name": "baseline",
        "learning_rate": 1e-3,
        "dropout": 0.30,
        "augment": False,
    },
    {
        **common,
        "name": "lower_learning_rate",
        "learning_rate": 3e-4,
        "dropout": 0.30,
        "augment": False,
    },
    {
        **common,
        "name": "augmentation",
        "learning_rate": 1e-3,
        "dropout": 0.30,
        "augment": True,
    },
]

if QUICK_RUN:
    experiments = experiments[:1]

trained_models = {}
histories = {}
summaries = []

for config in experiments:
    print("\n" + "=" * 70)
    print("Experiment:", config["name"])
    trained_model, history, summary = train_model(config)
    trained_models[config["name"]] = trained_model
    histories[config["name"]] = history
    summaries.append(summary)

print("\nValidation results")
for summary in summaries:
    print(summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in histories.items():
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    axes[0].plot(epochs, history["train_loss"], label=f"{name} train")
    axes[0].plot(epochs, history["val_loss"], linestyle="--", label=f"{name} val")
    axes[1].plot(epochs, history["train_accuracy"], label=f"{name} train")
    axes[1].plot(epochs, history["val_accuracy"], linestyle="--", label=f"{name} val")

axes[0].set(title="Loss during training", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[1].set(title="Accuracy during training", xlabel="Epoch", ylabel="Accuracy")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


## 8. Select the model and evaluate the test set once

Do not run this section until the full validation experiments are complete. The model is selected by the lowest validation loss; no test result influences the selection.


In [ ]:
if QUICK_RUN:
    raise RuntimeError(
        "Stop here: set QUICK_RUN = False, restart the kernel and run all cells "
        "before evaluating the test set."
    )

best_summary = min(summaries, key=lambda item: item["best_val_loss"])
best_name = best_summary["name"]
best_model = trained_models[best_name]
print("Selected model:", best_name)
print("Validation summary:", best_summary)


def predict(model, loader):
    model.eval()
    all_targets = []
    all_predictions = []
    all_probabilities = []
    with torch.no_grad():
        for inputs, targets in loader:
            logits = model(inputs.to(DEVICE))
            probabilities = torch.softmax(logits, dim=1).cpu()
            all_targets.append(targets)
            all_predictions.append(probabilities.argmax(dim=1))
            all_probabilities.append(probabilities)
    return (
        torch.cat(all_targets).numpy(),
        torch.cat(all_predictions).numpy(),
        torch.cat(all_probabilities).numpy(),
    )


_, _, test_loader = make_loaders(batch_size=64)
test_targets, test_predictions, test_probabilities = predict(best_model, test_loader)

test_accuracy = (test_predictions == test_targets).mean()
test_balanced_accuracy = balanced_accuracy_score(test_targets, test_predictions)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test balanced accuracy: {test_balanced_accuracy:.4f}")
print()
print(classification_report(test_targets, test_predictions, target_names=CLASS_NAMES, digits=4))


In [ ]:
cm = confusion_matrix(test_targets, test_predictions)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=ax, cmap="Blues", xticks_rotation=40, colorbar=False
)
ax.set_title(f"BloodMNIST test confusion matrix: {best_name}")
fig.tight_layout()
plt.show()

recall_per_class = np.diag(cm) / cm.sum(axis=1)
for class_name, recall in zip(CLASS_NAMES, recall_per_class):
    print(f"{class_name:24s} recall = {recall:.4f}")


## 9. Qualitative error analysis

The examples below show confident mistakes. Discuss whether confusing classes have similar colour, cell size, nucleus shape or texture. Avoid claiming a medical explanation that is not supported by literature.


In [ ]:
incorrect = np.flatnonzero(test_predictions != test_targets)
confidence = test_probabilities.max(axis=1)
most_confident_errors = incorrect[np.argsort(confidence[incorrect])[::-1][:16]]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, index in zip(axes.flat, most_confident_errors):
    ax.imshow(X_test_raw[index])
    true_name = CLASS_NAMES[test_targets[index]]
    predicted_name = CLASS_NAMES[test_predictions[index]]
    ax.set_title(
        f"True: {true_name}\nPred: {predicted_name} ({confidence[index]:.2f})",
        fontsize=8,
    )
    ax.axis("off")
fig.suptitle("Most confident incorrect test predictions")
fig.tight_layout()
plt.show()


## 10. Save reproducible outputs

Save the chosen weights and numerical results. Figures used in the report can be saved with `fig.savefig(...)` immediately before `plt.show()`.


In [ ]:
OUTPUT_DIR = Path("Assignment_2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

torch.save(best_model.state_dict(), OUTPUT_DIR / f"{best_name}_weights.pt")
results = {
    "seed": SEED,
    "device": str(DEVICE),
    "quick_run": QUICK_RUN,
    "selected_model": best_name,
    "validation": best_summary,
    "test_accuracy": float(test_accuracy),
    "test_balanced_accuracy": float(test_balanced_accuracy),
    "class_recall": {
        name: float(value) for name, value in zip(CLASS_NAMES, recall_per_class)
    },
    "experiments": summaries,
}
with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2)

print("Saved outputs to", OUTPUT_DIR.resolve())


## 11. Notes for the report

Report the full configuration and all validation experiments. State clearly that the official split was preserved and that model selection used validation data. Compare the final test result with the fully connected network from Assignment 1 and with the MedMNIST benchmark only after checking that the image resolution and evaluation metric are comparable.

Do not describe a quick run as the final experiment. Set `QUICK_RUN = False`, restart the kernel, run all cells, and preserve the outputs used in the report.
